# 01 — FAISS baseline từ CLIPFeatures BTC phát sẵn

**Phụ trách:** Tuấn Anh (N1) · **Mốc:** M1 · **GPU:** KHÔNG cần (CPU đủ)

Đầu vào: thư mục CLIPFeatures của BTC trên Drive chung.
Đầu ra: `faiss_clip.index` + `keyframe_ids.npy` lưu về Drive.

> Mở notebook này từ GitHub: Colab → File → Open notebook → GitHub → dán URL repo.

In [1]:
# === SETUP: mount Drive + cài thư viện ===
from google.colab import drive
drive.mount('/content/drive')
%pip -q install faiss-cpu numpy pandas

# === CONFIG: sửa đường dẫn theo Drive chung của đội ===
DRIVE_ROOT = '/content/drive/MyDrive/AIC2026'   # thư mục chia sẻ chung
CLIP_DIR   = f'{DRIVE_ROOT}/data/clip_features'  # nơi để CLIPFeatures BTC phát
OUT_DIR    = f'{DRIVE_ROOT}/outputs/index_v1_clip'
import os; os.makedirs(OUT_DIR, exist_ok=True)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 16.1 MB/s eta 0:00:00


In [ ]:
import os, glob
BASE = '/content/drive/MyDrive/AIC2026/data'
OUT  = '/content/drive/MyDrive/AIC2026/outputs'

def tim(ten):                      # tự gỡ trường hợp thư mục lồng 1 cấp
    d = os.path.join(BASE, ten)
    if glob.glob(f'{d}/*.npy') or glob.glob(f'{d}/*.csv'): return d
    for sub in sorted(glob.glob(f'{d}/*')):
        if os.path.isdir(sub) and (glob.glob(f'{sub}/*.npy') or glob.glob(f'{sub}/*.csv')):
            return sub
    return d

CLIP_DIR, MAP_DIR = tim('clip_features'), tim('map-keyframes')
os.makedirs(OUT, exist_ok=True)
print('CLIP_DIR =', CLIP_DIR, f'({len(glob.glob(f"{CLIP_DIR}/*.npy"))} file .npy)')
print('MAP_DIR  =', MAP_DIR,  f'({len(glob.glob(f"{MAP_DIR}/*.csv"))} file .csv)')

In [ ]:
import os, glob
dupes = sorted(glob.glob(f'{MAP_DIR}/* (*).csv'))
for d in dupes:
    goc = d.replace(' (1)', '')
    print(f'{os.path.basename(d):26} | có bản gốc: {os.path.exists(goc)}')

# Xem cột "có bản gốc" toàn True thì bỏ dấu # ở 2 dòng dưới rồi chạy lại
# for d in dupes: os.remove(d)
# print('Còn lại:', len(glob.glob(f'{MAP_DIR}/*.csv')), 'file — phải bằng 873')

In [ ]:
import os, glob, json, numpy as np, pandas as pd, faiss

vid_npy = {os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f'{CLIP_DIR}/*.npy')}
vid_csv = {os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f'{MAP_DIR}/*.csv')}
print(f'.npy: {len(vid_npy)} | .csv: {len(vid_csv)} | dùng chung: {len(vid_npy & vid_csv)}')

all_vecs, all_ids, lech = [], [], []
for i, vid in enumerate(sorted(vid_npy & vid_csv)):
    if i % 100 == 0: print(f'  ...{i}')
    vecs = np.load(f'{CLIP_DIR}/{vid}.npy').astype('float32')
    ns = pd.read_csv(f'{MAP_DIR}/{vid}.csv', usecols=['n'])['n'].tolist()
    if len(ns) != len(vecs):
        lech.append((vid, len(vecs), len(ns))); continue
    all_vecs.append(vecs)
    all_ids += [f'{vid}#{n:06d}' for n in ns]

X = np.concatenate(all_vecs)
faiss.normalize_L2(X)
index = faiss.IndexFlatIP(X.shape[1]); index.add(X)
faiss.write_index(index, f'{OUT}/clip_flat.faiss')
json.dump(all_ids, open(f'{OUT}/ids.json', 'w'))

print(f'\nXONG — {index.ntotal} vector, {X.shape[1]} chiều, {len(lech)} video lệch')
print('ID đầu:', all_ids[0], '  <-- phải kết thúc bằng #000001')

In [ ]:
import faiss, json
index = faiss.read_index(f'{OUT}/clip_flat.faiss')
all_ids = json.load(open(f'{OUT}/ids.json'))
print(index.ntotal, 'vector —', len(all_ids), 'id')

In [ ]:
%pip -q install open_clip_torch torch

In [ ]:
import faiss, torch, numpy as np, open_clip
from PIL import Image
from IPython.display import display

KF_DIR = '/content/drive/MyDrive/AIC2026/data/keyframes'
q = 'a news anchor in a television studio'   # đổi câu tùy thích

def show_top(model_name, k=5):
    model, _, _ = open_clip.create_model_and_transforms(model_name, pretrained='openai')
    tok = open_clip.get_tokenizer(model_name)
    with torch.no_grad():
        tv = model.encode_text(tok([q])).float().numpy()
    faiss.normalize_L2(tv)
    D, I = index.search(tv, k)
    print(f'\n=== {model_name} — "{q}" ===')
    for d, i in zip(D[0], I[0]):
        kf = all_ids[i]; vid, n = kf.split('#')
        print(f'{d:.3f}  {kf}')
        try:
            display(Image.open(f'{KF_DIR}/{vid}/{int(n):03d}.jpg').resize((320, 180)))
        except FileNotFoundError:
            print('   (chưa thấy ảnh — kiểm tra tên file keyframe)')

show_top('ViT-B-32-quickgelu')
show_top('ViT-B-16')